# Machine Learning Basics

This notebook introduces common machine learning methods with short theory notes and runnable code.

We use small, self-contained datasets so the examples run quickly and consistently.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_diabetes, load_breast_cancer, make_blobs
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score, f1_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, LogisticRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR, SVC
from sklearn.neighbors import KNeighborsRegressor
from sklearn.naive_bayes import GaussianNB
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA

try:
    import xgboost as xgb
except ImportError:
    xgb = None

sns.set_theme(style="whitegrid", context="notebook")
np.random.seed(7)

# Regression dataset
regression = load_diabetes()
X_reg = regression.data
y_reg = regression.target

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.25, random_state=7
)

# Classification dataset
classification = load_breast_cancer()
X_clf = classification.data
y_clf = classification.target

X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_clf, test_size=0.25, random_state=7, stratify=y_clf
)

# Simple clustering dataset
X_cluster, _ = make_blobs(n_samples=300, centers=4, random_state=7, cluster_std=1.2)

## 1. Linear Regression

Linear regression fits a straight-line relationship between inputs and the target. It minimizes the sum of squared errors and is a baseline for many regression problems.

In [ ]:
model = LinearRegression()
model.fit(X_train_reg, y_train_reg)
preds = model.predict(X_test_reg)

print("MAE:", mean_absolute_error(y_test_reg, preds))
print("R2:", r2_score(y_test_reg, preds))

## 2. Ridge Regression

Ridge adds an L2 penalty to shrink coefficients and reduce overfitting when features are correlated.

In [ ]:
model = Ridge(alpha=1.0)
model.fit(X_train_reg, y_train_reg)
preds = model.predict(X_test_reg)

print("MAE:", mean_absolute_error(y_test_reg, preds))
print("R2:", r2_score(y_test_reg, preds))

## 3. Lasso Regression

Lasso adds an L1 penalty that can push some coefficients to zero, which makes it useful for feature selection.

In [ ]:
model = Lasso(alpha=0.05, max_iter=5000)
model.fit(X_train_reg, y_train_reg)
preds = model.predict(X_test_reg)

print("Nonzero coefficients:", np.sum(model.coef_ != 0))
print("MAE:", mean_absolute_error(y_test_reg, preds))

## 4. Elastic Net

Elastic Net blends L1 and L2 penalties to balance sparsity and stability.

In [ ]:
model = ElasticNet(alpha=0.05, l1_ratio=0.5, max_iter=5000)
model.fit(X_train_reg, y_train_reg)
preds = model.predict(X_test_reg)

print("MAE:", mean_absolute_error(y_test_reg, preds))
print("R2:", r2_score(y_test_reg, preds))

## 5. Decision Tree Regression

Decision trees split the data into regions and fit a constant value in each region. They can model nonlinear relationships but can overfit without constraints.

In [ ]:
model = DecisionTreeRegressor(max_depth=4, random_state=7)
model.fit(X_train_reg, y_train_reg)
preds = model.predict(X_test_reg)

print("MAE:", mean_absolute_error(y_test_reg, preds))
print("R2:", r2_score(y_test_reg, preds))

## 6. Random Forest Regression

Random forests average many decision trees to reduce variance and improve generalization.

In [ ]:
model = RandomForestRegressor(n_estimators=200, random_state=7)
model.fit(X_train_reg, y_train_reg)
preds = model.predict(X_test_reg)

print("MAE:", mean_absolute_error(y_test_reg, preds))
print("R2:", r2_score(y_test_reg, preds))

## 7. Gradient Boosting Regression

Gradient boosting builds trees sequentially, with each new tree correcting errors from the previous ones.

In [ ]:
model = GradientBoostingRegressor(random_state=7)
model.fit(X_train_reg, y_train_reg)
preds = model.predict(X_test_reg)

print("MAE:", mean_absolute_error(y_test_reg, preds))
print("R2:", r2_score(y_test_reg, preds))

## 8. Support Vector Regression (SVR)

SVR finds a function that keeps most predictions within a small error margin, and can model nonlinear data with kernels.

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("svr", SVR(C=10.0, epsilon=5.0))
])
model.fit(X_train_reg, y_train_reg)
preds = model.predict(X_test_reg)

print("MAE:", mean_absolute_error(y_test_reg, preds))
print("R2:", r2_score(y_test_reg, preds))

## 9. K-Nearest Neighbors Regression

KNN predicts a value based on the average of the closest training points in feature space.

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", KNeighborsRegressor(n_neighbors=8))
])
model.fit(X_train_reg, y_train_reg)
preds = model.predict(X_test_reg)

print("MAE:", mean_absolute_error(y_test_reg, preds))
print("R2:", r2_score(y_test_reg, preds))

## 10. Logistic Regression

Logistic regression models the probability of a binary class using a sigmoid function. It is a strong baseline for classification problems.

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(max_iter=2000))
])
model.fit(X_train_clf, y_train_clf)
preds = model.predict(X_test_clf)

print("Accuracy:", accuracy_score(y_test_clf, preds))
print("F1:", f1_score(y_test_clf, preds))

In [ ]:
model = GaussianNB()
model.fit(X_train_clf, y_train_clf)
preds = model.predict(X_test_clf)

print("Accuracy:", accuracy_score(y_test_clf, preds))
print("F1:", f1_score(y_test_clf, preds))

In [ ]:
if xgb is None:
    print("xgboost is not installed. Install with: pip install xgboost")
else:
    model = xgb.XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=7,
    )
    model.fit(X_train_reg, y_train_reg)
    preds = model.predict(X_test_reg)

    print("MAE:", mean_absolute_error(y_test_reg, preds))
    print("R2:", r2_score(y_test_reg, preds))

## 13. XGBoost (Gradient-Boosted Trees)

XGBoost is a high-performance gradient-boosted tree model. It often delivers strong predictive accuracy by combining many shallow trees and regularization.

## 12. Naive Bayes

Naive Bayes classifiers assume features are conditionally independent given the class. They are fast, work well with high-dimensional data, and often provide strong baselines for classification.

## 14. K-Means Clustering

K-Means groups points into k clusters by minimizing within-cluster distances. It is an unsupervised method for discovering structure.

In [ ]:
model = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC(C=2.0, gamma="scale"))
])
model.fit(X_train_clf, y_train_clf)
preds = model.predict(X_test_clf)

print("Accuracy:", accuracy_score(y_test_clf, preds))
print("F1:", f1_score(y_test_clf, preds))

## 15. Principal Component Analysis (PCA)

PCA is a dimensionality reduction technique that finds new axes (components) that capture the most variance in the data.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)
kmeans = KMeans(n_clusters=4, random_state=7)
labels = kmeans.fit_predict(X_scaled)

plt.figure(figsize=(6, 5))
plt.scatter(X_scaled[:, 0], X_scaled[:, 1], c=labels, cmap="viridis", alpha=0.8)
plt.title("K-Means Clustering")
plt.xlabel("Feature 1 (scaled)")
plt.ylabel("Feature 2 (scaled)")
plt.tight_layout()
plt.show()

## 13. Principal Component Analysis (PCA)

PCA is a dimensionality reduction technique that finds new axes (components) that capture the most variance in the data.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_reg)
pca = PCA(n_components=2, random_state=7)
components = pca.fit_transform(X_scaled)

plt.figure(figsize=(6, 5))
plt.scatter(components[:, 0], components[:, 1], c=y_reg, cmap="coolwarm", alpha=0.8)
plt.title("PCA Projection of the Regression Dataset")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()

print("Explained variance ratio:", pca.explained_variance_ratio_)